# Inspect the matched MEG recordings

Read the existing matching CSV; do not recompute demographic matching, fit models, or preprocess signals. Inspect **every stored condition**, including conditions not selected by the analysis pipeline.

The main outputs are source-space evidence, the original array dimensions, trial counts when identifiable, and sampling frequency derived from the time axis. MATLAB v7.3 files are inspected through HDF5 metadata, avoiding reads of the large signal arrays. Conventional MATLAB files require loading `OUT` into memory, one file at a time.

A field named `sources_ERFs` and matching source coordinates support a source-space interpretation; they do **not** establish the reconstruction algorithm, coordinate units, or whether values are signed/absolute. Those details require explicit metadata or the reconstruction script. A two-dimensional source × time array does not reveal how many trials contributed to an average.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'vs0_load_and_config.py').exists():
    raise FileNotFoundError('Run this notebook from the iEEGvsMEG project directory.')
sys.path.insert(0, str(ROOT))
from vs0_load_and_config import MATCHED_PAIRS_CSV, read_matched_pairs

CSV_PATH = MATCHED_PAIRS_CSV  # Change only if the CSV is stored elsewhere.
TIME_UNIT = 's'  # Verify against export metadata; OUT.time is assumed to be seconds.
MAX_FILES = None  # None inspects all matched participants; use 1 for a first inspection.
ALLOW_CONVENTIONAL_MAT_LOAD = True  # False skips non-v7.3 files rather than loading OUT.
EXPORT_RESULTS = True
OUTPUT_DIR = ROOT / 'out' / 'meg_data_inspection'


## File availability

Paths are taken directly from `path_to_meg`; relative paths are resolved against the CSV directory. Missing files are recorded rather than replaced with other recordings.


In [ ]:
pairs = read_matched_pairs(CSV_PATH)
files = pairs[['ieeg_subject', 'meg_subject', 'path_to_meg']].copy()
files['exists'] = files.path_to_meg.map(lambda p: Path(p).is_file())
files['size_GiB'] = files.path_to_meg.map(lambda p: Path(p).stat().st_size / 2**30 if Path(p).is_file() else np.nan)
display(files)
print(f'{files.exists.sum()} / {len(files)} matched MEG files are accessible.')


## Inspection helpers

For MATLAB HDF5 datasets, dimensions are converted from HDF5 storage order to MATLAB order. Cell references are followed to inspect their target dimensions without reading signal values. Singleton cell wrappers are traversed; multiple arrays within a condition are reported separately.

Source and time axes must be uniquely identifiable from the coordinate count and time-vector length. Ambiguous layouts remain unresolved. For a three-dimensional source × time × trial array, the remaining axis gives the number of stored trials. This is not necessarily the original acquired trial count. Condition numbers are **one-based storage indices**, not inferred experimental labels.


In [ ]:
def time_summary(time, unit='s'):
    scales = {'s': 1., 'ms': .001}
    if unit not in scales:
        raise ValueError("TIME_UNIT must be 's' or 'ms'.")
    t = np.asarray(time, dtype=float).ravel() * scales[unit]
    if len(t) < 2 or not np.isfinite(t).all() or not np.all(np.diff(t) > 0):
        raise ValueError('Time vector must be finite, strictly increasing, and have at least two samples.')
    dt = np.diff(t)
    step = float(np.median(dt))
    regular = bool(np.allclose(dt, step, rtol=1e-4, atol=1e-10))
    return dict(n_times=len(t), tmin_s=float(t[0]), tmax_s=float(t[-1]),
                sampling_frequency_hz=1. / step if regular else np.nan,
                median_interval_s=step, uniform_time_axis=regular,
                max_interval_deviation_s=float(np.max(np.abs(dt-step))),
                assumed_time_unit=unit)


def interpret_shape(shape, n_sources, n_times):
    shape = tuple(int(v) for v in shape)
    result = dict(array_shape=str(shape), source_axis=None, time_axis=None,
                  trial_axis=None, n_trials=np.nan, source_axis_matches_coordinates=False,
                  layout_status='unresolved layout')
    if n_sources is None:
        result['layout_status'] = 'no usable source coordinate array'
        return result
    source_axes = [i for i, size in enumerate(shape) if size == n_sources]
    time_axes = [i for i, size in enumerate(shape) if size == n_times]
    candidates = [(a, b) for a in source_axes for b in time_axes if a != b]
    if len(candidates) != 1:
        result['layout_status'] = 'ambiguous or incompatible source/time axes'
        return result
    a, b = candidates[0]
    result.update(source_axis=a, time_axis=b, source_axis_matches_coordinates=True)
    remaining = [i for i in range(len(shape)) if i not in (a, b)]
    if len(shape) == 3:
        axis = remaining[0]
        result.update(trial_axis=axis, n_trials=shape[axis],
                      layout_status='trial count inferred from the third axis; verify exporter convention')
    elif len(shape) == 2:
        result['layout_status'] = 'source x time only; contributing trial count unavailable'
    else:
        result['layout_status'] = 'source/time axes identified; remaining axes require metadata'
    return result


def coordinate_count(shape):
    shape = tuple(shape)
    if len(shape) != 2:
        return None
    if shape[1] == 3:
        return int(shape[0])
    if shape[0] == 3 and shape[1] != 3:
        return int(shape[1])
    return None


def matlab_class(dataset):
    value = dataset.attrs.get('MATLAB_class', b'')
    return value.decode(errors='replace') if isinstance(value, bytes) else str(value)


def inspect_hdf5(path, unit):
    import h5py
    with h5py.File(path, 'r') as handle:
        def dereference(node):
            while isinstance(node, h5py.Dataset) and h5py.check_dtype(ref=node.dtype) is not None and node.size == 1:
                reference = np.asarray(node[()]).ravel()[0]
                if not reference:
                    raise ValueError(f'Empty MATLAB reference: {node.name}')
                node = handle[reference]
            return node

        def shape_of(node):
            # MATLAB v7.3 uses reversed dimensions; generic HDF5 does not.
            shape = tuple(node.shape)
            return shape[::-1] if 'MATLAB_class' in node.attrs else shape

        def small_values(node):
            node = dereference(node)
            if not isinstance(node, h5py.Dataset) or h5py.check_dtype(ref=node.dtype) is not None:
                raise ValueError('Expected a numeric metadata dataset.')
            if node.size > 2_000_000:
                raise ValueError(f'Metadata unexpectedly large: {node.name}; refusing a full read.')
            a = np.asarray(node[()])
            return a.transpose() if 'MATLAB_class' in node.attrs and a.ndim > 1 else a

        out = dereference(handle['OUT'])
        if not isinstance(out, h5py.Group):
            raise ValueError('OUT is not a scalar MATLAB structure/group.')
        inventory = []
        def visit(name, node):
            if name.count('/') > 3:
                return
            inventory.append(dict(field='OUT/' + name, storage_kind=type(node).__name__,
                matlab_class=matlab_class(node), storage_shape=str(node.shape) if isinstance(node, h5py.Dataset) else '',
                matlab_shape=str(shape_of(node)) if isinstance(node, h5py.Dataset) else '',
                dtype=str(node.dtype) if isinstance(node, h5py.Dataset) else ''))
        out.visititems(visit)
        info = time_summary(small_values(out['time']), unit)
        declared = {}
        for key in ('fsample', 'sfreq', 'fs', 'Fs', 'sampling_frequency'):
            if key in out:
                value = small_values(out[key])
                if value.size == 1:
                    declared[key] = float(value.ravel()[0])
        info['declared_sampling_frequency_fields'] = json.dumps(declared)
        info['declared_frequency_agrees_with_time'] = (
            all(np.isclose(v, info['sampling_frequency_hz'], rtol=1e-4) for v in declared.values())
            if declared and info['uniform_time_axis'] else None)

        coord = dereference(out['pos_brainsources_MNI8']) if 'pos_brainsources_MNI8' in out else None
        n_sources = coordinate_count(shape_of(coord)) if isinstance(coord, h5py.Dataset) else None
        source = out['sources_ERFs'] if 'sources_ERFs' in out else None
        if source is None:
            return info, n_sources, [], inventory, False

        def leaves(node, label):
            if isinstance(node, h5py.Dataset) and h5py.check_dtype(ref=node.dtype) is not None:
                refs = np.asarray(node[()])
                if 'MATLAB_class' in node.attrs and refs.ndim > 1:
                    refs = refs.transpose()
                result = []
                for i, ref in enumerate(refs.ravel(order='F'), 1):
                    if ref:
                        result.extend(leaves(handle[ref], f'{label}/{i}'))
                return result
            if not isinstance(node, h5py.Dataset):
                raise ValueError(f'Unexpected signal object: {node.name}')
            return [dict(array_location=label, **interpret_shape(shape_of(node), n_sources, info['n_times']))]

        rows = []
        # Do not unwrap the outer condition cell: even a single condition is meaningful.
        if isinstance(source, h5py.Dataset) and h5py.check_dtype(ref=source.dtype) is not None:
            refs = np.asarray(source[()])
            if 'MATLAB_class' in source.attrs and refs.ndim > 1:
                refs = refs.transpose()
            if sum(size > 1 for size in refs.shape) > 1:
                raise ValueError(f'Outer condition cell is a matrix {refs.shape}; determine its axes from the exporter.')
            for condition, ref in enumerate(refs.ravel(order='F'), 1):
                if ref:
                    rows.extend(dict(condition_index=condition, **row)
                                for row in leaves(handle[ref], f'sources_ERFs/{condition}'))
        else:
            rows.extend(dict(condition_index=1, **row) for row in leaves(source, 'sources_ERFs'))
        return info, n_sources, rows, inventory, True


def inspect_conventional_mat(path, unit):
    from scipy.io import loadmat
    # Preserve singleton dimensions to avoid erasing a one-trial axis.
    raw = loadmat(path, variable_names=['OUT'], struct_as_record=False, squeeze_me=False)
    out = raw['OUT'].item()
    fields = out._fieldnames
    inventory = [dict(field='OUT/' + name, storage_kind=type(getattr(out, name)).__name__,
                      matlab_class='', storage_shape=str(np.shape(getattr(out, name))),
                      matlab_shape=str(np.shape(getattr(out, name))),
                      dtype=str(getattr(getattr(out, name), 'dtype', ''))) for name in fields]
    info = time_summary(out.time, unit)
    declared = {}
    for key in ('fsample', 'sfreq', 'fs', 'Fs', 'sampling_frequency'):
        if key in fields:
            value = np.asarray(getattr(out, key))
            if value.size == 1 and value.dtype.kind in 'iuf':
                declared[key] = float(value.ravel()[0])
    info['declared_sampling_frequency_fields'] = json.dumps(declared)
    info['declared_frequency_agrees_with_time'] = (
        all(np.isclose(v, info['sampling_frequency_hz'], rtol=1e-4) for v in declared.values())
        if declared and info['uniform_time_axis'] else None)

    n_sources = coordinate_count(np.shape(out.pos_brainsources_MNI8)) if 'pos_brainsources_MNI8' in fields else None
    if 'sources_ERFs' not in fields:
        return info, n_sources, [], inventory, False

    def leaves(value, label):
        if isinstance(value, np.ndarray) and value.dtype == object:
            result = []
            for i, child in enumerate(value.ravel(order='F'), 1):
                result.extend(leaves(child, f'{label}/{i}'))
            return result
        a = np.asarray(value)
        if a.dtype.kind not in 'biufc':
            raise ValueError(f'Non-numeric signal array at {label}.')
        return [dict(array_location=label, **interpret_shape(a.shape, n_sources, info['n_times']))]

    source = out.sources_ERFs
    if source.dtype == object and sum(size > 1 for size in source.shape) > 1:
        raise ValueError(f'Outer condition cell is a matrix {source.shape}; determine its axes from the exporter.')
    conditions = source.ravel(order='F') if source.dtype == object else [source]
    rows = []
    for condition, value in enumerate(conditions, 1):
        rows.extend(dict(condition_index=condition, **row)
                    for row in leaves(value, f'sources_ERFs/{condition}'))
    return info, n_sources, rows, inventory, True


def inspect_file(path, unit='s', allow_conventional=True):
    import h5py
    path = Path(path)
    if path.suffix.lower() != '.mat':
        raise ValueError('This notebook inspects MATLAB recordings; unsupported file type: ' + path.suffix)
    is_hdf5 = h5py.is_hdf5(path)
    if not is_hdf5 and not allow_conventional:
        raise ValueError('Conventional MAT file skipped: ALLOW_CONVENTIONAL_MAT_LOAD=False.')
    info, n_sources, rows, inventory, source_field = (
        inspect_hdf5(path, unit) if is_hdf5 else inspect_conventional_mat(path, unit))
    evidence = ('sources_ERFs and compatible source-coordinate dimensions'
                if source_field and n_sources is not None and rows and all(r['source_axis_matches_coordinates'] for r in rows)
                else 'source fields present but dimensions require review' if source_field
                else 'sources_ERFs absent; inspect field inventory')
    info.update(file_format='MATLAB v7.3/HDF5' if is_hdf5 else 'conventional MATLAB',
                n_sources=n_sources, source_space_evidence=evidence,
                reconstruction_algorithm='not established by array structure',
                n_stored_conditions=len({r['condition_index'] for r in rows}))
    return info, rows, inventory


## Inspect accessible recordings

Files are inspected sequentially. Errors are retained in the file summary; other files continue. For a large conventional MAT file, allow enough memory or set `ALLOW_CONVENTIONAL_MAT_LOAD=False`.


In [ ]:
selected = files if MAX_FILES is None else files.iloc[:MAX_FILES]
file_rows, condition_rows, inventory_rows = [], [], []
for row in selected.itertuples(index=False):
    identity = dict(ieeg_subject=row.ieeg_subject, meg_subject=row.meg_subject, path_to_meg=row.path_to_meg)
    print(f'Inspecting {row.meg_subject}: {row.path_to_meg}', flush=True)
    try:
        info, conditions, inventory = inspect_file(row.path_to_meg, TIME_UNIT, ALLOW_CONVENTIONAL_MAT_LOAD)
        file_rows.append(dict(**identity, status='ok', error='', **info))
        condition_rows.extend(dict(**identity, **entry) for entry in conditions)
        inventory_rows.extend(dict(meg_subject=row.meg_subject, **entry) for entry in inventory)
    except Exception as exc:
        file_rows.append(dict(**identity, status='error', error=f'{type(exc).__name__}: {exc}'))

file_summary = pd.DataFrame(file_rows)
condition_summary = pd.DataFrame(condition_rows)
field_inventory = pd.DataFrame(inventory_rows)
display(file_summary)
if not condition_summary.empty:
    display(condition_summary)
else:
    print('No condition arrays were identified. Review file errors and field inventory.')


## Answers and consistency checks

Trial counts are shown separately for each stored condition and nested array; they are not summed across potentially different representations. If multiple arrays occur within a condition, inspect their locations before deciding which is the trial-level signal. Verify the experimental names corresponding to the condition indices using the field inventory or exporter.

The sampling-frequency estimate is $f_s = 1 / \operatorname{median}(t_{k+1}-t_k)$ after converting time values to seconds. A nonuniform time axis is flagged and no single sampling frequency is reported. The time-unit setting must be verified; the numerical time vector alone cannot establish its units.


In [ ]:
if 'status' in file_summary and file_summary.status.eq('ok').any():
    successful = file_summary.loc[file_summary.status.eq('ok')]
    display(successful[['meg_subject', 'source_space_evidence', 'n_sources', 'n_stored_conditions',
                        'sampling_frequency_hz', 'uniform_time_axis', 'assumed_time_unit']])
    print('Sampling frequencies (Hz):', successful.sampling_frequency_hz.dropna().round(6).unique())
    print('Reconstruction algorithm: not established unless confirmed in the metadata or exporter.')
if not condition_summary.empty:
    display(condition_summary[['meg_subject', 'condition_index', 'array_location', 'array_shape',
                               'n_trials', 'layout_status']])
    known = condition_summary.dropna(subset=['n_trials'])
    if not known.empty:
        display(known.groupby('condition_index').n_trials.agg(['count', 'min', 'max']).rename(columns={'count': 'n_arrays'}))
    if condition_summary.n_trials.isna().any():
        print('Some trial counts are unavailable or ambiguous; see layout_status.')
if not field_inventory.empty:
    display(field_inventory)


In [ ]:
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    files.to_csv(OUTPUT_DIR / 'file_availability.csv', index=False)
    file_summary.to_csv(OUTPUT_DIR / 'meg_file_summary.csv', index=False)
    condition_summary.to_csv(OUTPUT_DIR / 'meg_conditions.csv', index=False)
    field_inventory.to_csv(OUTPUT_DIR / 'meg_field_inventory.csv', index=False)
    (OUTPUT_DIR / 'inspection_settings.json').write_text(json.dumps(dict(
        csv_path=str(Path(CSV_PATH).resolve()), time_unit=TIME_UNIT, max_files=MAX_FILES,
        allow_conventional_mat_load=ALLOW_CONVENTIONAL_MAT_LOAD), indent=2))
    print('Saved inspection tables:', OUTPUT_DIR.resolve())
